In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# 경로 확인 (이전에 변경한 최종 파일명 반영 권장)
DB_PATH = 'database/ESG_final.db' 

def run_outlier_detection_and_update():
    # 1. 임계치 설정
    L2_LIMITS = {
        'Site A': {'electricity_mwh': 3348.0, 'lng_nm3': 100000.0},
        'Site B': {'electricity_mwh': 5000.0, 'lng_nm3': 88704.0}
    }
    L1_Z_THRESHOLD, L1_YOY_THRESHOLD, L3_INTENSITY_THRESHOLD = 3.0, 30.0, 50.0 

    # 2. 데이터 처리 (Connection 안전하게 관리)
    with sqlite3.connect(DB_PATH) as conn:
        usage_df = pd.read_sql("SELECT * FROM standard_usage", conn)
        activity_df = pd.read_sql("SELECT * FROM activity_data", conn)
        
        for site in usage_df['site_id'].unique():
            for metric in usage_df['metric'].unique():
                # 데이터 정렬 및 병합
                site_usage = usage_df[(usage_df['site_id'] == site) & (usage_df['metric'] == metric)].sort_values('reporting_date')
                site_act = activity_df[activity_df['site_id'] == site].sort_values('reporting_date')
                merged = pd.merge(site_usage, site_act, on=['site_id', 'reporting_date'])
                
                # 검증 루프 (13개월차부터)
                for i in range(12, len(merged)):
                    row = merged.iloc[i]
                    val, std_id = row['value'], int(row['id'])
                    
                    # [L1~L3 탐지 알고리즘]
                    window = merged.iloc[i-12:i]['value']
                    z_score = abs(val - window.mean()) / window.std() if window.std() > 0 else 0
                    yoy_roc = abs((val - merged.iloc[i-12]['value']) / merged.iloc[i-12]['value'] * 100)
                    limit = L2_LIMITS[site][metric]
                    l2_fail = val > limit
                    intensity = val / row['production_qty']
                    hist_int = (merged.iloc[i-12:i]['value'] / merged.iloc[i-12:i]['production_qty']).mean()
                    dev = abs((intensity - hist_int) / hist_int * 100)
                    
                    # [상태 업데이트 판정]
                    if (z_score > L1_Z_THRESHOLD) or (yoy_roc > L1_YOY_THRESHOLD) or l2_fail or (dev > L3_INTENSITY_THRESHOLD):
                        # 2번: 이상치(Anomaly)
                        conn.execute("UPDATE standard_usage SET v_status = 2 WHERE id = ?", (std_id,))
                        
                        layers = []
                        if (z_score > L1_Z_THRESHOLD) or (yoy_roc > L1_YOY_THRESHOLD): layers.append(f"L1(Z:{z_score:.1f})")
                        if l2_fail: layers.append(f"L2(Limit:{limit})")
                        if dev > L3_INTENSITY_THRESHOLD: layers.append(f"L3(Dev:{dev:.1f}%)")
                        
                        severity = "Critical" if l2_fail else "Major" if dev > L3_INTENSITY_THRESHOLD else "Warning"
                        
                        conn.execute("""
                            INSERT INTO outlier_results (std_id, layer, detected_value, threshold, severity)
                            VALUES (?, ?, ?, ?, ?)
                        """, (std_id, ", ".join(layers), val, limit, severity))
                    else:
                        # 1번: 정상(Pass)
                        conn.execute("UPDATE standard_usage SET v_status = 1 WHERE id = ?", (std_id,))
                        
    print("✅ 이상치 탐지 엔진 실행 및 넘버링(1:정상, 2:이상치) 업데이트가 완료되었습니다.")

if __name__ == "__main__":
    run_outlier_detection_and_update()